### Evaluation of SUTVA pass/fail/inconclusive rate across all real datasets.

In its very strict version:

In [4]:
def check_strict_sutva(dataset_description, variables_summary, llm):
    """SUTVA check with strict prompt — flags theoretical spillovers."""
    from cais.methods.pre_model_assumption_utils import _llm_argue_assumption
    return _llm_argue_assumption(
        assumption_name="SUTVA (Stable Unit Treatment Value Assumption)",
        assumption_description=(
            "(1) No interference: one unit's treatment does not affect another unit's "
            "potential outcomes. (2) No hidden versions of the treatment: the treatment "
            "is administered consistently across treated units."
        ),
        dataset_description=dataset_description,
        variables_summary=variables_summary,
        llm=llm,
        extra_context=(
            "Pay attention to: network/spillover effects (e.g., units in shared "
            "schools, households, markets), partial compliance, treatment intensity "
            "variation."
        ),
    )

In [5]:
import pandas as pd
from cais.config import get_llm_client

info = pd.read_csv('../data/real_info.csv', encoding='utf-8-sig')

# some of the datasets appear several times, deleting duplicates
datasets = info.drop_duplicates(subset='data_files')[['data_files', 'data_description', 'method',
                                                       'treatment', 'outcome', 'covariates',
                                                       'instrument_var', 'temporal_var']].reset_index(drop=True)

print(f"Number of unique datasets: {len(datasets)}\n")

Number of unique datasets: 19



In [6]:
llm = get_llm_client()

results = []

for i, row in datasets.iterrows():
    dataset_name = row['data_files']
    description = row['data_description']

    variables_summary = {
        'treatment': row['treatment'],
        'outcome': row['outcome'],
        'covariates': row['covariates'],
        'method': row['method'],
        'instrument': row.get('instrument_var', None),
        'time_variable': row.get('temporal_var', None),
    }

    print(f"[{i+1}/{len(datasets)}] {dataset_name}...", end=" ")

    strict_result = check_strict_sutva(description, variables_summary, llm=llm)

    status = {True: 'PASS', False: 'FAIL', None: 'INCONCLUSIVE'}[strict_result['passed']]
    print(status)

    results.append({
        'dataset': dataset_name,
        'method': row['method'],
        'passed': strict_result['passed'],
        'status': status,
        'reasoning': strict_result['reasoning'],
        'missing_info': strict_result.get('details', {}).get('missing_info', None),
    })

[1/19] voter_turnout_data.csv... FAIL
[2/19] lalonde_data.csv... PASS
[3/19] lalonde_data_psid.csv... PASS
[4/19] vernby_2019.csv... PASS
[5/19] card_geographic.csv... INCONCLUSIVE
[6/19] lee_2008.csv... PASS
[7/19] abortion_bf15.csv... FAIL
[8/19] abortion_bm15.csv... FAIL
[9/19] broockman_intrinsic.csv... PASS
[10/19] castle.csv... FAIL
[11/19] gov_transfers.csv... FAIL
[12/19] organ_donations.csv... PASS
[13/19] thornton_hiv.csv... PASS
[14/19] close_elections.csv... FAIL
[15/19] electrification_data.csv... FAIL
[16/19] nan... FAIL
[17/19] fda_carpenter.csv... PASS
[18/19] fulton.csv... FAIL
[19/19] hansen.csv... PASS


In [7]:
strict_result_df = pd.DataFrame(results)

# SUTVA SUMMARY ON REAL DATASETS

print(f"\n{strict_result_df['status'].value_counts().to_string()}")
print(f"\nPASS Rate: {(strict_result_df['status']=='PASS').mean():.0%}")
print(f"FAIL Rate: {(strict_result_df['status']=='FAIL').mean():.0%}")
print(f"INCONCLUSIVE Rate: {(strict_result_df['status']=='INCONCLUSIVE').mean():.0%}")


status
FAIL            9
PASS            9
INCONCLUSIVE    1

PASS Rate: 47%
FAIL Rate: 47%
INCONCLUSIVE Rate: 5%


In [8]:
# Detail
print(strict_result_df[['dataset', 'method', 'status', 'reasoning']].to_string(index=False))

                 dataset   method       status                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                reasoning
  voter_turnout_data.csv      ols         FAIL                                                                                          The SUTVA assumption is likely violated due to potential interference among individuals within the same 

In [9]:
strict_df = pd.DataFrame(strict_result_df)

fails = strict_df[strict_df['status'] == 'FAIL']
print(f"FAIL number (strict) : {len(fails)} / {len(strict_df)}\n")

for _, row in fails.iterrows():
    print(f"  Dataset : {row['dataset']}")
    print(f"  Method  : {row['method']}")
    print(f"  Status  : FAIL")
    print(f"  Reasoning :")
    print(f"    {row['reasoning']}")
    if row.get('missing_info'):
        print(f"  Missing info : {row['missing_info']}")
    print(f"\n")

FAIL number (strict) : 9 / 19

  Dataset : voter_turnout_data.csv
  Method  : ols
  Status  : FAIL
  Reasoning :
    The SUTVA assumption is likely violated due to potential interference among individuals within the same household. For example, the treatment groups that emphasize social pressure (like Neighbors and Self) could lead to discussions or influence among household members, affecting each other's voting behavior. Additionally, the presence of multiple individuals in a household could create variations in treatment exposure and response, further complicating the assumption of no interference. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.


  Dataset : abortion_bf15.csv
  Method  : did
  Status  : FAIL
  Reasoning :
    The assumption of no interference is likely violated in this context, as the legalization of abortion in one state could influence behaviors and health outcomes in neighboring 

In its permissive version:

In [10]:
import pandas as pd
from cais.methods.pre_model_assumption_utils import check_sutva
from cais.config import get_llm_client

info = pd.read_csv('../data/real_info.csv', encoding='utf-8-sig')

datasets = info.drop_duplicates(subset='data_files')[['data_files', 'data_description', 'method',
                                                       'treatment', 'outcome', 'covariates',
                                                       'instrument_var', 'temporal_var']].reset_index(drop=True)

print(f"Number of unique datasets: {len(datasets)}\n")

Number of unique datasets: 19



In [11]:
llm = get_llm_client()

results = []

for i, row in datasets.iterrows():
    dataset_name = row['data_files']
    description = row['data_description']

    variables_summary = {
        'treatment': row['treatment'],
        'outcome': row['outcome'],
        'covariates': row['covariates'],
        'method': row['method'],
        'instrument': row.get('instrument_var', None),
        'time_variable': row.get('temporal_var', None),
    }

    print(f"[{i+1}/{len(datasets)}] {dataset_name}...", end=" ")

    permissive_result = check_sutva(description, variables_summary, llm=llm)

    status = {True: 'PASS', False: 'FAIL', None: 'INCONCLUSIVE'}[permissive_result['passed']]
    print(status)

    results.append({
        'dataset': dataset_name,
        'method': row['method'],
        'passed': permissive_result['passed'],
        'status': status,
        'reasoning': permissive_result['reasoning'],
        'missing_info': permissive_result.get('details', {}).get('missing_info', None),
    })

[1/19] voter_turnout_data.csv... PASS
[2/19] lalonde_data.csv... PASS
[3/19] lalonde_data_psid.csv... PASS
[4/19] vernby_2019.csv... PASS
[5/19] card_geographic.csv... PASS
[6/19] lee_2008.csv... PASS
[7/19] abortion_bf15.csv... PASS
[8/19] abortion_bm15.csv... PASS
[9/19] broockman_intrinsic.csv... PASS
[10/19] castle.csv... PASS
[11/19] gov_transfers.csv... PASS
[12/19] organ_donations.csv... PASS
[13/19] thornton_hiv.csv... PASS
[14/19] close_elections.csv... PASS
[15/19] electrification_data.csv... PASS
[16/19] nan... PASS
[17/19] fda_carpenter.csv... PASS
[18/19] fulton.csv... PASS
[19/19] hansen.csv... PASS


In [12]:
permissive_result_df = pd.DataFrame(results)

# SUTVA SUMMARY ON REAL DATASETS

print(f"\n{permissive_result_df['status'].value_counts().to_string()}")
print(f"\nPASS Rate: {(permissive_result_df['status']=='PASS').mean():.0%}")
print(f"FAIL Rate: {(permissive_result_df['status']=='FAIL').mean():.0%}")
print(f"INCONCLUSIVE Rate: {(permissive_result_df['status']=='INCONCLUSIVE').mean():.0%}")


status
PASS    19

PASS Rate: 100%
FAIL Rate: 0%
INCONCLUSIVE Rate: 0%


In [13]:
# Detail
print(permissive_result_df[['dataset', 'method', 'status', 'reasoning']].to_string(index=False))

                 dataset   method status                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          reasoning
  voter_turnout_data.csv      ols   PASS                                                                                                                The design of the experiment involves random assignment of households to treatment groups, which minimizes the risk of interference 

In [14]:
permissive_df = pd.DataFrame(permissive_result_df)

passes = permissive_df[permissive_df['status'] == 'PASS']
print(f"PASS Number (permissive) : {len(passes)} / {len(permissive_df)}\n")

for _, row in passes.iterrows():
    print(f"  Dataset : {row['dataset']}")
    print(f"  Method  : {row['method']}")
    print(f"  Status  : PASS")
    print(f"  Reasoning :")
    print(f"    {row['reasoning']}")
    if row.get('missing_info'):
        print(f"  Missing info : {row['missing_info']}")
    print(f"\n")

PASS Number (permissive) : 19 / 19

  Dataset : voter_turnout_data.csv
  Method  : ols
  Status  : PASS
  Reasoning :
    The design of the experiment involves random assignment of households to treatment groups, which minimizes the risk of interference between units. Each treatment group received a distinct mailing that was consistent across participants, suggesting that there are no hidden versions of the treatment. Given the nature of the mailings and the focus on individual households, the assumption of no interference is reasonably satisfied. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.


  Dataset : lalonde_data.csv
  Method  : ols
  Status  : PASS
  Reasoning :
    The NSW Demonstration involved random assignment of participants to treatment and control groups, which minimizes the risk of interference between units. Additionally, the treatment (employment program) was consistently administered

On a study we fore sure know it doesn't satisfy SUTVA: Twenty Year Economic Effects of Deworming, Miguel & Kremer (2004)

In [15]:
df = pd.read_stata('../data/all_data/Worm_Infection_Panel.dta')
print(df.shape)
print(df.columns.tolist())

(2695, 29)
['pupid', 'klps_popweight', 'year', 'female', 'male', 'older', 'younger', 'treat_1999', 'treat_2001', 'psdpsch98', 'pup_pop', 'avgtest96', 'popT_6k', 'zoneidI2', 'zoneidI3', 'zoneidI4', 'zoneidI5', 'zoneidI6', 'zoneidI7', 'zoneidI8', 'std98_base_I2', 'std98_base_I3', 'std98_base_I4', 'std98_base_I5', 'std98_base_I6', 'any_moderate_who_1999', 'z_intensity_who_1999', 'any_moderate_who_2001', 'z_intensity_who_2001']


In [16]:
deworming_description2 = (
    "Miguel & Kremer (2004) — School-based deworming program in Kenya. "
    "Children in 75 primary schools were randomly assigned to early or late treatment groups. "
    "Treatment consisted of mass deworming medication (albendazole + praziquantel). "
    "CRITICAL CONTEXT: Worm infections are transmitted between children through shared "
    "environments (soil, water). Treating children in one school reduces parasite prevalence "
    "for untreated children in the same school AND in nearby schools (within 3km). "
    "The original paper explicitly models and estimates these spillover/externality effects. "
)

deworming_description3 = (
    "Miguel & Kremer (2004) — School-based deworming program in Kenya. "
    "Children in 75 primary schools were randomly assigned to early or late treatment groups. "
    "Treatment consisted of mass deworming medication (albendazole + praziquantel). "
    "Worm infections are transmitted between children through shared environments (soil, water)."
)


deworming_description = (
    "Estimating the impact of child health investments on adult living standards entails multiple "
    "methodological challenges, including the lack of experimental variation in health status, "
    "an inability to track individuals over time, and accurately measuring living standards "
    "and productivity in low-income settings. This study exploits a randomized school health "
    "intervention that provided deworming treatment to Kenyan children, and uses longitudinal data "
    "to estimate impacts on economic outcomes up to 20 years later. The effective respondent tracking "
    "rate was 84%. Individuals who received two to three additional years of childhood deworming "
    "experienced a 14% gain in consumption expenditures and 13% increase in hourly earnings. "
    "There are also shifts in sectors of residence and employment: Treatment group individuals "
    "are 9% more likely to live in urban areas, and experience a 9% increase in nonagricultural "
    "work hours. Most effects are concentrated among males and older individuals. The observed "
    "consumption and earnings benefits, together with deworming’s low cost when distributed at "
    "scale, imply that a conservative estimate of its annualized social internal rate of return "
    "is 37%, a high return by any standard."
)


deworming_variables = {
    'treatment': 'deworming treatment',
    'outcome': 'consumption expenditures, hourly earnings',
    'unit': 'individual children in schools',
    'method': 'RCT',
}

In [17]:
# Strict version
result_strict = check_strict_sutva(deworming_description, deworming_variables, llm=llm)
print("STRICT :", result_strict)

STRICT : {'passed': True, 'reasoning': 'The SUTVA assumption appears to be satisfied in this study as the randomized controlled trial design minimizes the likelihood of interference between individuals, given that deworming treatment is administered at the school level. Additionally, the treatment (deworming) is consistently applied across treated individuals, with no indication of hidden versions of the treatment affecting the outcomes. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [18]:
# Strict version
result_strict = check_strict_sutva(deworming_description2, deworming_variables, llm=llm)
print("STRICT :", result_strict)

STRICT : {'passed': False, 'reasoning': 'The SUTVA assumption is not plausibly satisfied in this context due to the presence of interference among units. Since worm infections can be transmitted between children through shared environments, treating children in one school can affect the health outcomes of untreated children in the same school and nearby schools. This indicates that the treatment of one unit (child) can influence the potential outcomes of another unit, violating the no interference condition of SUTVA. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [19]:
# Strict version
result_strict = check_strict_sutva(deworming_description3, deworming_variables, llm=llm)
print("STRICT :", result_strict)

STRICT : {'passed': False, 'reasoning': "The assumption of no interference is likely violated in this context because worm infections can be transmitted between children through shared environments, meaning that one child's treatment could affect the health outcomes of others in the same school. Additionally, the treatment is administered to groups of children, which raises concerns about consistent treatment effects across individuals. Therefore, the SUTVA assumption is not plausibly satisfied. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.", 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [20]:
# Permissive version
result_permissive = check_sutva(deworming_description, deworming_variables, llm=llm)
print("PERMISSIVE :", result_permissive)

PERMISSIVE : {'passed': True, 'reasoning': 'The study employs a randomized controlled trial (RCT) design to evaluate the effects of deworming treatment on economic outcomes, which typically supports the SUTVA assumption. The treatment is administered consistently to individuals in schools, and there is no indication of interference among treated and untreated individuals, as the outcomes measured (consumption expenditures and hourly earnings) are individual-level metrics. Additionally, the study does not suggest any significant spillover effects or variations in treatment application. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [21]:
# Permissive version
result_permissive = check_sutva(deworming_description2, deworming_variables, llm=llm)
print("PERMISSIVE :", result_permissive)

PERMISSIVE : {'passed': False, 'reasoning': 'The assumption of no interference is violated in this context because worm infections are transmitted between children through shared environments, meaning that treating children in one school can affect the health outcomes of untreated children in the same school and nearby schools. This creates a clear mechanism for spillover effects, which the original paper explicitly models and estimates. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


In [22]:
# Permissive version
result_permissive = check_sutva(deworming_description3, deworming_variables, llm=llm)
print("PERMISSIVE :", result_permissive)

PERMISSIVE : {'passed': False, 'reasoning': "The assumption of no interference is violated in this context because worm infections can be transmitted between children through shared environments, such as soil and water. This suggests that one child's treatment could affect another child's potential outcomes, particularly in a school setting where children interact closely. Additionally, the treatment is not administered uniformly across all children, as some receive early treatment while others receive late treatment, which could lead to variations in treatment effects. Note: this assessment relies on LLM reasoning and is sensitive to the quality and completeness of the dataset description provided.", 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


**Conclusion**:

The LLM-based SUTVA check is **only as good as the dataset description provided**.
When the description omits key mechanistic details (e.g., transmission pathways, shared environments), the check may return false negatives even on well-known SUTVA violations such as Miguel & Kremer (2004).

## On synthetic data

In [23]:
info = pd.read_csv('../data/synthetic_info.csv', encoding='utf-8-sig')

print(f"Number of synthetic datasets : {len(info)}")

llm = get_llm_client()

results = []

for i, row in info.iterrows():
    dataset_name = row['data_files']
    description = row['data_description']
    method = row['method']

    # No columns treatment/outcome/covariates separated in synthetic_info,
    # everything is in data_description
    variables_summary = {
        'method': method,
        'query': row['natural_language_query'],
    }

    print(f"[{i+1}/{len(info)}] {dataset_name} ({method})...", end=" ")

    result = check_sutva(description, variables_summary, llm=llm)

    status = {True: 'PASS', False: 'FAIL', None: 'INCONCLUSIVE'}[result['passed']]
    print(status)

    results.append({
        'dataset': dataset_name,
        'method': method,
        'passed': result['passed'],
        'status': status,
        'reasoning': result['reasoning'],
        'missing_info': result.get('details', {}).get('missing_info', None),
    })

results_df = pd.DataFrame(results)


# SUTVA SUMMARY ON SYNTHETIC DATASETS

print(f"\nGlobal :")
print(results_df['status'].value_counts().to_string())
print(f"\nPASS Rate: {(results_df['status']=='PASS').mean():.0%}")
print(f"FAIL Rate: {(results_df['status']=='FAIL').mean():.0%}")
print(f"INCONCLUSIVE Rate: {(results_df['status']=='INCONCLUSIVE').mean():.0%}")

print(f"\nPer method:")
print(results_df.groupby('method')['status'].value_counts().unstack(fill_value=0).to_string())

Number of synthetic datasets : 45
[1/45] did_canonical_data_0.csv (did_canonical)... PASS
[2/45] did_canonical_data_1.csv (did_canonical)... PASS
[3/45] did_canonical_data_3.csv (did_canonical)... PASS
[4/45] did_canonical_data_2.csv (did_canonical)... PASS
[5/45] did_canonical_data_4.csv (did_canonical)... PASS
[6/45] did_twfe_data_4.csv (did_twfe)... PASS
[7/45] did_twfe_data_2.csv (did_twfe)... PASS
[8/45] did_twfe_data_3.csv (did_twfe)... PASS
[9/45] did_twfe_data_1.csv (did_twfe)... PASS
[10/45] did_twfe_data_0.csv (did_twfe)... PASS
[11/45] iv_encouragement_data_4.csv (iv_encouragement)... PASS
[12/45] iv_encouragement_data_0.csv (iv_encouragement)... FAIL
[13/45] iv_encouragement_data_1.csv (iv_encouragement)... PASS
[14/45] iv_encouragement_data_3.csv (iv_encouragement)... PASS
[15/45] iv_encouragement_data_2.csv (iv_encouragement)... PASS
[16/45] iv_data_0.csv (iv)... PASS
[17/45] iv_data_1.csv (iv)... PASS
[18/45] iv_data_3.csv (iv)... PASS
[19/45] iv_data_2.csv (iv)... PASS


In [24]:
# Detail
for _, row in results_df.iterrows():
    print(f"\n  {row['dataset']} ({row['method']}) → {row['status']}")
    print(f"    {row['reasoning'][:150]}...")


  did_canonical_data_0.csv (did_canonical) → PASS
    The dataset describes a study conducted in public schools where the treatment (biannual health check-ups) is applied uniformly across students in scho...

  did_canonical_data_1.csv (did_canonical) → PASS
    The dataset describes a study of factories where the treatment (adoption of the industrial reform policy) is applied at the factory level, and there i...

  did_canonical_data_3.csv (did_canonical) → PASS
    The dataset involves multiple schools with unique identifiers, and the treatment (tutoring initiative) is applied at the school level, suggesting that...

  did_canonical_data_2.csv (did_canonical) → PASS
    The dataset describes a government program providing free solar panels to selected households, suggesting a clear treatment assignment without interfe...

  did_canonical_data_4.csv (did_canonical) → PASS
    The dataset involves retail stores in two different states, with a clear distinction between those affected b

In [25]:
results_strict = []

for i, row in info.iterrows():
    dataset_name = row['data_files']
    description = row['data_description']
    method = row['method']

    variables_summary = {
        'method': method,
        'query': row['natural_language_query'],
    }

    print(f"[{i+1}/{len(info)}] {dataset_name} ({method})...", end=" ")

    result = check_strict_sutva(description, variables_summary, llm=llm)

    status = {True: 'PASS', False: 'FAIL', None: 'INCONCLUSIVE'}[result['passed']]
    print(status)

    results_strict.append({
        'dataset': dataset_name,
        'method': method,
        'passed': result['passed'],
        'status': status,
        'reasoning': result['reasoning'],
        'missing_info': result.get('details', {}).get('missing_info', None),
    })

results_strict_df = pd.DataFrame(results_strict)


# SUTVA SUMMARY ON SYNTHETIC DATASETS

print(f"\nGlobal :")
print(results_strict_df['status'].value_counts().to_string())
print(f"\nPASS Rate: {(results_strict_df['status']=='PASS').mean():.0%}")
print(f"FAIL Rate: {(results_strict_df['status']=='FAIL').mean():.0%}")
print(f"INCONCLUSIVE Rate: {(results_strict_df['status']=='INCONCLUSIVE').mean():.0%}")

print(f"\nPer method:")
print(results_strict_df.groupby('method')['status'].value_counts().unstack(fill_value=0).to_string())

[1/45] did_canonical_data_0.csv (did_canonical)... FAIL
[2/45] did_canonical_data_1.csv (did_canonical)... FAIL
[3/45] did_canonical_data_3.csv (did_canonical)... FAIL
[4/45] did_canonical_data_2.csv (did_canonical)... FAIL
[5/45] did_canonical_data_4.csv (did_canonical)... FAIL
[6/45] did_twfe_data_4.csv (did_twfe)... FAIL
[7/45] did_twfe_data_2.csv (did_twfe)... FAIL
[8/45] did_twfe_data_3.csv (did_twfe)... FAIL
[9/45] did_twfe_data_1.csv (did_twfe)... FAIL
[10/45] did_twfe_data_0.csv (did_twfe)... FAIL
[11/45] iv_encouragement_data_4.csv (iv_encouragement)... FAIL
[12/45] iv_encouragement_data_0.csv (iv_encouragement)... FAIL
[13/45] iv_encouragement_data_1.csv (iv_encouragement)... PASS
[14/45] iv_encouragement_data_3.csv (iv_encouragement)... FAIL
[15/45] iv_encouragement_data_2.csv (iv_encouragement)... FAIL
[16/45] iv_data_0.csv (iv)... FAIL
[17/45] iv_data_1.csv (iv)... FAIL
[18/45] iv_data_3.csv (iv)... FAIL
[19/45] iv_data_2.csv (iv)... PASS
[20/45] iv_data_4.csv (iv)... FAIL

In [26]:
# Detail
for _, row in results_strict_df.iterrows():
    print(f"\n  {row['dataset']} ({row['method']}) → {row['status']}")
    print(f"    {row['reasoning'][:150]}...")


  did_canonical_data_0.csv (did_canonical) → FAIL
    The assumption of no interference is likely violated in this study because students within the same school may influence each other's health behaviors...

  did_canonical_data_1.csv (did_canonical) → FAIL
    The assumption of no interference is likely violated in this context, as factories may influence each other's production outputs through shared labor ...

  did_canonical_data_3.csv (did_canonical) → FAIL
    The assumption of no interference is likely violated in this context, as schools may influence each other's outcomes through shared students, resource...

  did_canonical_data_2.csv (did_canonical) → FAIL
    The assumption of no interference is likely violated in this context, as households in rural areas may share resources or influence each other's energ...

  did_canonical_data_4.csv (did_canonical) → FAIL
    The assumption of no interference is likely violated in this context, as retail stores in close proximity may